In [1]:
import pandas as pd
import geopandas as gpd
import json
from functools import partial
from tqdm import tqdm

import matplotlib.pyplot as plt

import nomad.io.base as loader
import nomad.stop_detection.utils as utils
from nomad.stop_detection.density_based import seqscan_labels
from nomad.stop_detection.dbscan import ta_dbscan_labels

import nomad.visit_attribution.visit_attribution as visits

from nomad.stop_detection.validation import (
    AlgorithmRegistry,
    compute_stop_detection_metrics,
    plot_metric_vs_param,
 )
import warnings
warnings.filterwarnings('ignore', message='Input is timezone-naive; assuming UTC')

## Load Files

In [20]:
key = '2_stops'
with open(f'config_{key}.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

out = config['output_files']

poi_table = gpd.read_parquet(config['buildings_file']).rename(columns={'id': 'location'})
sparse_df = loader.from_file(out['sparse_path'], format='parquet')
sparse_df.drop(columns=['datetime'], inplace=True)
diaries_df = loader.from_file(out['diaries_path'], format='parquet').rename(
    columns={'identifier': 'user_id'})
poi_subset = poi_table.loc[poi_table.location.isin(diaries_df['location'].dropna())]

# R stores all parameterized algorithm runs and can be iterated directly in execution.
R = AlgorithmRegistry()
R.add_algorithm(
    seqscan_labels,
    family='seqscan',
    granularity=10,
    time_thresh=60,
    dist_thresh=(6, 25),
    min_pts=2,
    back_merge=False,
    x='x',
    y='y',
    timestamp='timestamp',
)
R.add_algorithm(
    ta_dbscan_labels,
    family='st_dbscan',
    granularity=10,
    time_thresh=60,
    dist_thresh=(6, 25),
    min_pts=2,
    x='x',
    y='y',
    timestamp='timestamp',
)

summarize_stops_with_loc = partial(
    utils.summarize_stop,
    x='x',
    y='y',
    timestamp='timestamp',
    keep_col_names=True,
    passthrough_cols=['location', 'user_id'],
    complete_output=False)

## Execution of stop detection

In [ ]:
results_list = []
users = diaries_df.user_id.unique()

sparse_df['precomp_locations'] = visits.poi_map(
    sparse_df,
    poi_table=poi_subset,
    data_crs='EPSG:3857',
    max_distance=20,
    location_id='location',
    x='x',
    y='y',
)

stops_list = []
sparse_list = []
for algo in tqdm(R, desc='Processing algorithms'):
    for user in users:
        sparse = sparse_df.query("user_id==@user").copy()
        truth = diaries_df.query("user_id==@user").copy()

        if user == 'jovial_jennings':
            labels = R.time_call(algo, sparse)
            sparse['cluster'] = labels
            sparse['location'] = sparse['precomp_locations']

            sparse['location'] = visits.point_in_polygon(
                sparse,
                poi_table=poi_subset,
                method='majority',
                location_id='location',
                recompute_location=False,
                x='x',
                y='y',
            )

            stops = sparse[sparse.cluster != -1].groupby('cluster', as_index=False).apply(
                summarize_stops_with_loc, include_groups=False)

            
            stops['algorithm'] = algo['algorithm']
            stops['dist_thresh'] = algo['params']['dist_thresh']

            sparse['algorithm'] = algo['algorithm']
            sparse['dist_thresh'] = algo['params']['dist_thresh']
            stops_list = stops_list + [stops]
            sparse_list = sparse_list + [sparse]
            metrics = compute_stop_detection_metrics(
                stops=stops,
                truth=truth,
                user_id=user,
                algorithm=algo['algorithm'],
                traj_cols={'location_id': 'location'},
                timestamp='timestamp',
            )

            results_list.append(R.annotate_metrics(metrics, algo))
        else:
            continue
results_df = pd.DataFrame(results_list)
timing_summary = R.family_timing_summary()

timing_summary

Processing algorithms: 100%|██████████| 20/20 [00:19<00:00,  1.02it/s]


,family,n,mean_s,avg_pings
1,st_dbscan,10,0.008706,38.0
0,seqscan,10,0.018624,38.0


In [ ]:
# we probably want to switch to using nomad.base.io.to_file 
# we probably only want the cluster labels (and or locations) from all_sparse,
# since the trajectory itself is in the parquet file

all_stops = pd.concat(stops_list)
all_sparse = pd.concat(sparse_list)

all_stops.to_csv("sample_stops.csv", index=False)
all_sparse.to_csv("sample_labels.csv", index=False)